<a href="https://colab.research.google.com/github/kshgrshrn/VIMA3YA_assignment/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NimbusPay ticket triage

The default run retrains the selected adapter and writes predictions for all 400 test inputs. The full experiment matrix is optional and its saved results are included in the repository.

In [ ]:
# Install the pinned training libraries.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'peft==0.18.1', 'accelerate==1.12.0', 'bitsandbytes==0.49.0'])
print('Training libraries installed. Ready to upload the assignment and code.')

Training libraries installed. Ready to upload the assignment and code.


In [ ]:
from google.colab import files as colab_files
from pathlib import Path
from urllib.request import urlopen

print('Upload the original training_assessment.zip.')
uploaded = colab_files.upload()
if 'training_assessment.zip' not in uploaded:
    raise FileNotFoundError('Please upload training_assessment.zip.')
Path('/content/training_assessment.zip').write_bytes(uploaded['training_assessment.zip'])

project = Path('/content/nimbuspay')
repo = 'https://raw.githubusercontent.com/kshgrshrn/VIMA3YA_assignment/main/'
files_to_download = [
    'run_assessment.py', 'pipeline.py', 'cleaning.py', 'requirements.txt', 'final_config.json',
    'results/audit_summary.json', 'results/experiment_table.csv',
    'results/baseline_fixed/scores.json', 'results/baseline_rules/scores.json',
    'results/A_raw/scores.json', 'results/A_raw/training_metrics.json',
    'results/B_clean/scores.json', 'results/B_clean/training_metrics.json',
    'results/C_fp16/scores.json', 'results/C_fp16/training_metrics.json',
    'results/D_rank32/scores.json', 'results/D_rank32/training_metrics.json',
    'results/D_rank32/dev_predictions.jsonl',
    'results/E_boundary/scores.json', 'results/E_boundary/training_metrics.json',
]
for name in files_to_download:
    destination = project / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(repo + name, timeout=60) as response:
        destination.write_bytes(response.read())
print('Downloaded the runner and saved experiment metadata from this repository.')

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        '-u',
        '/content/nimbuspay/run_assessment.py',
        '--mode', 'smoke',
        '--archive', '/content/training_assessment.zip',
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/nimbuspay/run_assessment.py', '--mode', 'smoke', '--archive', '/content/training_assessment.zip'], returncode=0)

In [ ]:
# Optional: set True to rerun the full experiment matrix (about 79 minutes on the recorded T4).
import subprocess
import sys
RUN_FULL_EXPERIMENTS = False

if RUN_FULL_EXPERIMENTS:
    subprocess.run(
        [
            sys.executable,
            '-u',
            '/content/nimbuspay/run_assessment.py',
            '--mode', 'experiments',
            '--archive', '/content/training_assessment.zip',
        ],
        check=True,
    )
else:
    print('Skipped the full matrix. Saved results are in the repository results/ folder.')


In [ ]:
!python -u /content/nimbuspay/run_assessment.py \
  --mode reproduce \
  --archive /content/training_assessment.zip

AUDIT: {"raw_rows": 1445, "clean_rows": 1360, "dropped": 85, "fixed": 175, "findings": {"priority_rule_recomputed": 60, "legacy_priority_key_and_value": 70, "legacy_date_to_iso": 30, "invalid_json_label": 25, "normalize_transaction_id": 20, "rupees_stored_as_paise": 25, "exact_duplicate_input": 60}, "limitations": ["No automatic category/language relabelling", "No complete semantic date parser", "Ambiguous amount discrepancies are dropped, not guessed", "Regexes for repeated contact are not exhaustive; existing true labels retained"]}
ENVIRONMENT: {"python": "3.13.16", "gpu": "Tesla T4", "cuda": "13.0", "packages": {"torch": "2.14.1+cu130", "transformers": "4.57.6", "peft": "0.18.1", "accelerate": "1.12.0", "bitsandbytes": "0.49.0", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}}
LENGTHS {'min': 97, 'median': 168, 'p95': 225, 'p99': 1816, 'max': 2215, 'policy': 'No truncation; dynamic padding; microbatch 1 above 768 tokens.'}
I0000 00:00:1794478760.390296   30827 cpu_feature_guar